# Time series (PCMCI, PCMCI+, LPCMCI, Granger, VARMA-LiNGAM, longitudinal LiNGAM)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abel-ai-lab/andrey/blob/main/examples/time_series.ipynb)

Six methods find causes in data measured over time. PCMCI and Granger find the links from earlier
steps; PCMCI+ also finds links within one step, and LPCMCI allows series that are not measured.
VARMA-LiNGAM estimates the weights of the links within one step and from the step before.
Longitudinal LiNGAM takes a panel instead: the same units measured at a few time points. Each
returns a `TemporalStructure`, one graph per lag.

## Start

Choose a CPU runtime and run this cell first. In Colab it installs `andrey-core==0.10.0` from PyPI;
local kernels use the installed environment.

In [1]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    os.environ["ANDREY_DEVICE"] = "cpu"
    !pip install --quiet "andrey-core[viz]==0.10.0" pandas

In [2]:
import numpy as np
import pandas as pd
from IPython.display import display

import andrey
import andrey.viz
from andrey.data import sample_scm

## Data

`sample_scm` draws one series of 1,000 time steps over 4 variables: a vector autoregression with
one lag, links within each step from a random DAG, and uniform noise. The drawing merges the lags
into one graph; a loop is a variable's dependence on its own past.

In [3]:
series = sample_scm(
    graph="erdos_renyi",
    functional="var",
    noise="uniform",
    d=4,
    n=1000,
    seed=1,
    density=1.0,
    scale="raw",
)
X, truth = series.data, series.graph
print(f"{X.shape[0]} time steps of {X.shape[1]} variables")
print(truth)
display(andrey.viz.draw(truth.summary_graph()))

1000 time steps of 4 variables
temporal  |  4 nodes  |  2 lags

  lag 0  |  2 edges (2 directed)
    0 -> 1
    0 -> 3

  lag 1  |  10 edges (10 directed)
    0 -> 0
    0 -> 1
    1 -> 0
    0 -> 3
    1 -> 1
    2 -> 1
    1 -> 3
    2 -> 2
    2 -> 3
    3 -> 3


'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 360 240" width="100%" font-family="Inter, system-ui, sans-serif"><rect width="360" height="240" fill="#fafbfc"/><path d="M 76.00 57.00 C 67.00 41.00 93.00 41.00 84.00 57.00" fill="none" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="84.00,57.00 80.50,50.00 89.00,51.00" fill="#3a4658" stroke-linejoin="round"/><path d="M 90.73 77.34 Q 125.26 100.94 157.03 102.64" fill="none" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="167.02,103.17 157.30,97.65 156.77,107.63" fill="#3a4658" stroke-linejoin="round"/><path d="M 169.27 96.54 Q 134.74 72.94 102.97 71.23" fill="none" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="92.98,70.70 102.70,76.22 103.23,66.24" fill="#3a4658" stroke-linejoin="round"/><path d="M 92.99 70.52 Q 182.17 74.12 258.00 97.18" fill="none" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="267.56,100.09 259.45,92.39 256.54,101.96" fill="#3a4658" stroke-linejoin="round"/><path d="M 176.00 90.87 C 167.00 74.87 193.00 74.87 184.00 90.87" fill="none" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="184.00,90.87 180.50,83.87 189.00,84.87" fill="#3a4658" stroke-linejoin="round"/><line x1="160.82" y1="116.56" x2="90.84" y2="162.83" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="169.16,111.04 158.06,112.39 163.57,120.73" fill="#3a4658" stroke-linejoin="round"/><line x1="193.00" y1="103.87" x2="257.00" y2="103.87" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="267.00,103.87 257.00,98.87 257.00,108.87" fill="#3a4658" stroke-linejoin="round"/><path d="M 76.00 157.00 C 67.00 141.00 93.00 141.00 84.00 157.00" fill="none" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="84.00,157.00 80.50,150.00 89.00,151.00" fill="#3a4658" stroke-linejoin="round"/><line x1="92.34" y1="165.92" x2="258.16" y2="111.09" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="267.66,107.95 256.59,106.34 259.73,115.84" fill="#3a4658" stroke-linejoin="round"/><path d="M 276.00 90.87 C 267.00 74.87 293.00 74.87 284.00 90.87" fill="none" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="284.00,90.87 280.50,83.87 289.00,84.87" fill="#3a4658" stroke-linejoin="round"/><circle cx="80.00" cy="70.00" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="80.00" y="70.00" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">0</text><circle cx="180.00" cy="103.87" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="180.00" y="103.87" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">1</text><circle cx="80.00" cy="170.00" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="80.00" y="170.00" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">2</text><circle cx="280.00" cy="103.87" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="280.00" y="103.87" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">3</text></svg>'

## Links from the past: PCMCI and Granger

PCMCI tests each lagged link with partial correlations, given the strongest earlier causes it
finds first; `tau_max` is the largest lag tested. Granger regresses each variable on the past
`tau_max` steps of every variable with a cross-validated lasso. Neither tests links within one step.

In [4]:
print(andrey.pcmci(X, tau_max=1), end="\n\n")
print(andrey.granger(X, tau_max=1))

pcmci  temporal  |  4 nodes  |  1 lags

  lag 1  |  10 edges (10 directed)
    0 -> 0
    0 -> 1
    1 -> 0
    0 -> 3
    1 -> 1
    2 -> 1
    1 -> 3
    2 -> 2
    2 -> 3
    3 -> 3

  p_values: a list of 1, too long to print; read metadata['p_values']
  statistics: a list of 1, too long to print; read metadata['statistics']
  parents: [[[1, 1], [0, 1]], [[1, 1], [0, 1], [2, 1]], [[2, 1], [3, 1]], [[1, 1], [2, 1], [0, 1]]]

granger  temporal  |  4 nodes  |  1 lags

  lag 1  |  13 edges (13 directed)
    0 -> 0
    0 -> 1
    1 -> 0
    2 -> 0
    0 -> 3
    1 -> 1
    2 -> 1
    1 -> 3
    3 -> 1
    2 -> 2
    2 -> 3
    3 -> 2
    3 -> 3


## Links within a step: PCMCI+ and LPCMCI

PCMCI+ adds the same-step links and orients them where the data allow; lag 0 is a CPDAG. LPCMCI
allows hidden series, so it returns a partial ancestral graph (PAG): a circle marks an end the data
leave open.

In [5]:
print(andrey.pcmciplus(X, tau_max=1), end="\n\n")
print(andrey.lpcmci(X, tau_max=1))

pcmciplus  temporal  |  4 nodes  |  2 lags

  lag 0  |  3 edges (3 directed)
    0 -> 1
    3 -> 0
    1 -> 3

  lag 1  |  7 edges (7 directed)
    0 -> 0
    0 -> 1
    1 -> 0
    1 -> 1
    2 -> 1
    1 -> 3
    2 -> 2

  p_values: a list of 2, too long to print; read metadata['p_values']
  statistics: a list of 2, too long to print; read metadata['statistics']
  parents: [[[1, 1], [0, 1]], [[1, 1], [0, 1], [2, 1]], [[2, 1]], [[1, 1], [2, 1]]]
  ambiguous_triples: [[0, 1, 1, 3]]

lpcmci  temporal  |  4 nodes  |  2 lags

  lag 0  |  2 edges (1 partially_directed, 1 directed)
    0 o-> 1
    0 -> 3

  lag 1  |  0 edges (no edges)

  links: a list of 2, too long to print; read metadata['links']
  p_values: a list of 2, too long to print; read metadata['p_values']
  statistics: a list of 2, too long to print; read metadata['statistics']
  parents: [[], [], [], [[0, 0]]]


## Weights: VARMA-LiNGAM

VARMA-LiNGAM uses the non-Gaussian noise to order the variables within a step, and estimates the
weight of every link. `prune=True` sets small weights to zero.

The likelihood fit behind it stops at `max_iter` iterations, 100 by default, and warns when it
does. This series needs about 630 iterations to converge, so the cell raises the cap to 700; with
the default, the fit stops early, warns, and returns a different graph. The fit takes about two
minutes.

In [6]:
# The likelihood fit depends on the CPU's BLAS kernels, so on another type of CPU the graph
# can differ.
out = andrey.varma_lingam(X, order=(1, 1), prune=True, max_iter=700)
print(out, end="\n\n")
print("lag 0 weights, row = cause:")
print(np.round(out.structure.lag_weights[0], 2))

varma_lingam  temporal  |  4 nodes  |  2 lags

  lag 0  |  3 edges (3 directed)
    0 -> 1
    0 -> 3
    3 -> 1

  lag 1  |  12 edges (12 directed)
    0 -> 0
    1 -> 0
    2 -> 0
    0 -> 3
    3 -> 0
    1 -> 1
    2 -> 1
    1 -> 3
    3 -> 1
    2 -> 2
    2 -> 3
    3 -> 3

  order: 0 -> 3 -> 2 -> 1

lag 0 weights, row = cause:
[[ 0.    1.31  0.   -1.38]
 [ 0.    0.    0.    0.  ]
 [ 0.    0.    0.    0.  ]
 [ 0.    0.08  0.    0.  ]]


## Compare the links

Each method's links at lags 0 and 1 beside the true ones, a variable's dependence on its own past
included. A same-step link counts once, whatever its direction; a lagged link counts with its
direction. LPCMCI is left out: its lagged graphs keep only the links it orients as causes, so its
PAG above is the answer to read.

VARMA-LiNGAM's row is shown on its own, because its likelihood fit depends on the CPU's BLAS
kernels and its graph above and its counts can differ on another type of CPU.

In [7]:
def links(structure):
    found = set()
    for lag in (0, 1):
        if lag not in structure.lags:
            continue
        for i, j, _ in structure.lag(lag).oriented_edges():
            found.add((min(i, j), max(i, j), 0) if lag == 0 else (i, j, 1))
    return found


true_links = links(truth)


def row(fit):
    found = links(fit.structure)
    return {
        "true links found": f"{len(found & true_links)} of {len(true_links)}",
        "extra links": len(found - true_links),
    }


fits = {
    "PCMCI": andrey.pcmci(X, tau_max=1),
    "Granger": andrey.granger(X, tau_max=1),
    "PCMCI+": andrey.pcmciplus(X, tau_max=1),
}
pd.DataFrame({name: row(fit) for name, fit in fits.items()}).T

,true links found,extra links
PCMCI,10 of 12,0
Granger,10 of 12,3
PCMCI+,9 of 12,1


In [8]:
# The likelihood fit depends on the CPU's BLAS kernels, so on another type of CPU this row can
# differ.
pd.DataFrame({"VARMA-LiNGAM": row(out)}).T

,true links found,extra links
VARMA-LiNGAM,11 of 12,4


## Panels: longitudinal LiNGAM

Longitudinal LiNGAM takes one table per time point, the same units in the same rows. Here 500 units
are measured at 3 time points; within each, `x0 -> x1 -> x2`, and each variable depends on its own
value at the time point before. The columns come in one order drawn from the seed.

In [9]:
CHAIN = [("x0", "x1"), ("x1", "x2")]  # the true edges within a time point, cause first


def sample_panel(seed: int, n: int = 500, shuffle: bool = True) -> list[pd.DataFrame]:
    """Draw n units at 3 time points, the columns in one order drawn from the seed."""
    rng = np.random.default_rng(seed)
    b0 = np.array([[0.0, 0.0, 0.0], [0.8, 0.0, 0.0], [0.0, -0.7, 0.0]])  # row = effect
    mix = np.linalg.inv(np.eye(3) - b0)
    panel = [rng.uniform(-1, 1, (n, 3)) @ mix.T]
    for _ in range(2):
        panel.append((0.5 * panel[-1] + rng.uniform(-1, 1, (n, 3))) @ mix.T)
    names = ["x0", "x1", "x2"]
    columns = list(rng.permutation(names)) if shuffle else names
    return [pd.DataFrame(x, columns=names)[columns] for x in panel]


out = andrey.longitudinal_lingam(sample_panel(seed=0), tau_max=1)
print(out, end="\n\n")
print("lag 0 at the last time point:", out.structure.lag(0).oriented_edges())

longitudinal_lingam  temporal  |  3 nodes  |  2 lags

  lag 0  |  2 edges (2 directed)
    x0 -> x1
    x1 -> x2

  lag 1  |  9 edges (9 directed)
    x0 -> x0
    x0 -> x1
    x1 -> x0
    x0 -> x2
    x2 -> x0
    x1 -> x1
    x1 -> x2
    x2 -> x1
    x2 -> x2

  causal_orders: [[], [0, 1, 2], [0, 1, 2]]

lag 0 at the last time point: [('x0', 'x1', 'directed'), ('x1', 'x2', 'directed')]


The [release cards](https://andrey.abel.ai/docs/benchmarks.html) compare each method with tigramite,
lingam, or causal-learn on the simulations of the methods' papers.

## Use your own data

Pass a table with one row per time step, such as a DataFrame whose column names become the graph's
labels, or change `seed` and `n` above and rerun.